# GPT2 Text Generation with KerasNLP


Here, we take a look at how we can use a pretrained LLM model (GPT-2) to generate text. This is based on the tutorial [here](https://keras.io/examples/generative/gpt2_text_generation_with_kerasnlp/).

Here, we only focus on loading and using a pre-trained model. You can also [fine-tune the model](https://keras.io/examples/nlp/parameter_efficient_finetuning_of_gpt2_with_lora/) to specific text style, but we don't get into that. Moreover, you can also create a LLM text generator from scratch, which is a bit more involved. An example is provided [here](https://keras.io/examples/generative/text_generation_gpt/).

Note that the fine-tuning of the model and creating it from scratch requires use of powerful GPUs, otherwise, it cannot be done within a reasonable amount of time.

First, let's bring in the required dependencies. We can also choose the backend engine from `"tensorflow"`, `"jax"` or `"torch"`.


In [1]:
!pip install git+https://github.com/keras-team/keras-nlp.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.27.0.dev0 which is incompatible.


In [2]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"  # or "jax" or "torch"

import keras_nlp
import keras
import tensorflow as tf
import time

keras.mixed_precision.set_global_policy("mixed_float16")

Large Language Models are complex to build and expensive to train from scratch.
Luckily there are pretrained LLMs available for use right away. [KerasNLP](https://keras.io/keras_nlp/)
provides a large number of pre-trained checkpoints that allow you to experiment without needing to train them yourself, including pretrained models with `generate()` method such as `keras_nlp.models.GPT2CausalLM` and `keras_nlp.models.OPTCausalLM`.

## Load a pre-trained GPT-2 model and generate some text

KerasNLP provides a number of pre-trained models, such as [Google
Bert](https://ai.googleblog.com/2018/11/open-sourcing-bert-state-of-art-pre.html)
and [GPT-2](https://openai.com/research/better-language-models).

It's very easy to load the GPT-2 model as you can see below:

In [3]:
# To speed up training and generation, we use preprocessor of length 128
# instead of full length 1024.
preprocessor = keras_nlp.models.GPT2CausalLMPreprocessor.from_preset(
    "gpt2_base_en",
    sequence_length=128,
)
gpt2_lm = keras_nlp.models.GPT2CausalLM.from_preset(
    "gpt2_base_en", preprocessor=preprocessor
)

100%|██████████| 431/431 [00:00<00:00, 586kB/s]


100%|██████████| 618/618 [00:00<00:00, 1.67MB/s]


100%|██████████| 0.99M/0.99M [00:01<00:00, 719kB/s]


100%|██████████| 446k/446k [00:01<00:00, 400kB/s]


100%|██████████| 475M/475M [00:38<00:00, 13.0MB/s]


Once the model is loaded, you can use it to generate some text right away. Run
the cells below to give it a try. It's as simple as calling a single function
*generate()*:

In [4]:
start = time.time()

output = gpt2_lm.generate("My trip to Yosemite was", max_length=200)
print("\nGPT-2 output:")
print(output)

end = time.time()
print(f"TOTAL TIME ELAPSED: {end - start:.2f}s")


GPT-2 output:
My trip to Yosemite was the most beautiful day of my life. I was at my first place on my first day of work. The sun was just starting to set and it seemed like I had been on the trail for over an hour. The view was stunning and I could feel myself on the edge of my seat and my feet on the ground. The sun was setting so fast that I felt my legs start to shake. The sun was still setting but it was still bright enough for me to see my feet. The view was still so bright that I felt like I was in a dream. I could feel the earth and the sky. It was beautiful. I didn't know how to describe how beautiful this view was. My eyes were still on the sky and my body was still shaking. It was so beautiful. I felt like I was in the middle of a movie or a movie set. The sun was still shining and the sky was just beginning. I was just so happy and excited
TOTAL TIME ELAPSED: 16.38s


Try another one:

In [5]:
start = time.time()

output = gpt2_lm.generate("That data analytics course was", max_length=200)
print("\nGPT-2 output:")
print(output)

end = time.time()
print(f"TOTAL TIME ELAPSED: {end - start:.2f}s")


GPT-2 output:
That data analytics course was designed for the University of Chicago's Computer Science & Engineering Department.

In this course, we're going to take a look at the ways in which we can use the data analytics of the Internet, and how we might be able to use them to improve our performance.

We're looking at data analytics from the Internet, from mobile apps to the cloud, and from the cloud-based applications. We'll use the tools of Google Analytics for this course to analyze data, and we'll use Google's cloud-based data analytics to improve our performance.

In this course, we'll look at how we can use the data analytics of the Internet, from mobile apps to the cloud, and from the cloud-based applications. We'll use the tools of Google Analytics for this course to analyze data, and we'll use Google's cloud-based data analytics to improve our performance.

We're going to look at how we can use the data
TOTAL TIME ELAPSED: 1.76s


Notice how much faster the second call is. This is because the computational
graph is [XLA compiled](https://www.tensorflow.org/xla) in the 1st run and
re-used in the 2nd behind the scenes.

The quality of the generated text is hit and miss!

## Finetune on Chinese Poem Dataset

We can also finetune GPT2 on non-English datasets. For readers knowing Chinese,
this part illustrates how to fine-tune GPT2 on Chinese poem dataset to teach our
model to become a poet!

Because GPT2 uses byte-pair encoder, and the original pretraining dataset
contains some Chinese characters, we can use the original vocab to finetune on
Chinese dataset.

In [6]:
!# Load chinese poetry dataset.
!git clone https://github.com/chinese-poetry/chinese-poetry.git

Cloning into 'chinese-poetry'...
remote: Enumerating objects: 7341, done.
remote: Counting objects: 100% (1843/1843), done.
remote: Compressing objects: 100% (828/828), done.
remote: Total 7341 (delta 1055), reused 1015 (delta 1015), pack-reused 5498 (from 1)
Receiving objects: 100% (7341/7341), 236.94 MiB | 15.48 MiB/s, done.
Resolving deltas: 100% (5050/5050), done.
Updating files: 100% (2285/2285), done.


Load text from the json file. We only use《全唐诗》for demo purposes.

In [7]:
import os
import json

poem_collection = []
for file in os.listdir("chinese-poetry/全唐诗"):
    if ".json" not in file or "poet" not in file:
        continue
    full_filename = "%s/%s" % ("chinese-poetry/全唐诗", file)
    with open(full_filename, "r") as f:
        content = json.load(f)
        poem_collection.extend(content)

paragraphs = ["".join(data["paragraphs"]) for data in poem_collection]

Let's take a look at sample data.

In [12]:
print(paragraphs[:5])

['入朝貴國慙下客，七日承恩作上賓。更見鳯（一作「凰」）聲無妓態，風流變動一園（一作「國」）春。（此詩爲仁貞在日本國作。見金毓黻撰《渤海國志長編》卷十八引《文華秀麗集》上。）。', '不體（《類從》本作「航」。〖又日本佛教全書〗本作「{舟巳}」。考云：「{舟巳}即那字。慧超《傳》多用此字。」）心淚（《類從》本作「渡」。）自涓，（望按：「此句原脫一字。」）情因法眼奄幽泉。明朝儻問滄波客，的說遺鞋白足還。（見《渤海國志長編》卷十八引《入唐求法巡禮行記》三。）（〖1〗按詩前有小注，曰「詩一首。在唐作」。蓋從當時渤海藩國言，故曰「在唐」也。又詩末題「大和二年四月十四日書」十字。〖2〗上海古籍出版社一九八六年出版顧承甫等點校本《入唐求來巡禮行記》，參據多種版本校勘，較精審。今錄異文如次：「公作而習之」，「作」作「僕」；「余之身期降物」，作「余亦身期絳物」；「入宗五臺」，「宗」作「室」；「青癡」，作「青瘀」；「仙大師己來日久」，「己」作「亡」；「位我之血」，「位」作「泣」；「不體心淚」，「體」作「航」。）。', '天生石月架空虛，樹綴龍髯子貫珠。三十年前已攀折，建公曾到上方無。（見明陶宗儀《古刻叢鈔》。）（詩下原署曰「上都左街保壽寺文章應制內供奉大德元孚」。又《寶刻叢編》卷十五有《唐房田寺經藏院記》，云唐崔龜從撰，僧元孚書，會昌二年立。則元孚者，武宗提人也。）。', '。', '上德乘杯渡，金人道已東。戒香餘散馥，慧炬復流風。月隱歸靈鷲，珠逃入梵宮。神飛生死表，遺教法門中。（〖1〗見同前書《唐大和上東征傳》。詩題下原署曰「日本國傳燈沙門思託」。〖汪向榮校注本《唐大和上東征傳》錄此詩題作《五言傷大和上傳燈逝日本》，署「傳燈沙門釋思託」。以「日本」二字歸詩題，是，今據補。思託爲鑒真弟子，《東征傳》中多次提及。〗餘參閱前元開詩附注。）。']


First, we convert to TF dataset, and only use partial data
to train.

In [13]:
train_ds = (
    tf.data.Dataset.from_tensor_slices(paragraphs)
    .batch(16)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# Running through the whole dataset takes long, only take `200` and run 1
# epochs for demo purposes.
train_ds = train_ds.take(200)
num_epochs = 1

learning_rate = keras.optimizers.schedules.PolynomialDecay(
    5e-4,
    decay_steps=train_ds.cardinality() * num_epochs,
    end_learning_rate=0.0,
)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
gpt2_lm.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss=loss,
    weighted_metrics=["accuracy"],
)

gpt2_lm.fit(train_ds, epochs=num_epochs)

200/200 ━━━━━━━━━━━━━━━━━━━━ 161s 355ms/step - accuracy: 0.2581 - loss: 2.6038


Let's check the result!

In [14]:
output = gpt2_lm.generate("昨夜雨疏风骤", max_length=500)
print(output)

昨夜雨疏风骤，清清聲自自爲。還青堂非更晝，自細清聲風晴。深自堂塵坐賞，更須風爲須陳。賢致色紛風青


Apparently it is not too bad!!